**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# Session 27: Preference 데이터 수집/생성

## 🎯 선호도 데이터란?

DPO, RLHF 등 선호도 기반 학습에는 **"어떤 응답이 더 좋은지"** 비교 데이터가 필요합니다.  
이 노트북에서는 선호도(Preference) 데이터의 개념, 형식, 수집 방법, 그리고 자동 생성 방법을 학습합니다.

### 학습 목표
- 🎯 Preference 데이터의 구조와 역할 이해
- 1️⃣ Chosen/Rejected 데이터 형식 파악
- 2️⃣ 인간 평가 기반 데이터 수집 방법론 이해
- 3️⃣ GPT-4를 활용한 자동 선호도 데이터 생성 실습
- 4️⃣ 데이터 품질 검증 방법 학습

### GPU 요구사항
- ⭕ GPU 선택적 (Section 7에서 로컬 모델 사용 시 필요)
- 대부분의 실습은 OpenAI API 기반

---

In [ ]:
# 필요한 라이브러리 임포트
import json
import os
import random
from pathlib import Path
from datetime import datetime

print("✅ 기본 라이브러리 임포트 완료")
print(f"📅 실행 시각: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# OpenAI API 키 확인
api_key = os.getenv("OPENAI_API_KEY", "")
if api_key:
    print("✅ OPENAI_API_KEY 설정 확인됨")
else:
    print("⚠️ OPENAI_API_KEY가 설정되지 않았습니다.")
    print("   export OPENAI_API_KEY='your-key-here' 또는 아래 셀에서 직접 설정하세요.")

In [ ]:
# API 키 직접 설정 (필요 시)
# os.environ["OPENAI_API_KEY"] = "sk-your-key-here"

# 데이터 디렉토리 설정
DATA_DIR = Path("../data/samples")
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ 데이터 디렉토리: {DATA_DIR.resolve()}")
print(f"✅ 출력 디렉토리: {OUTPUT_DIR.resolve()}")

---
## 1️⃣ Chosen/Rejected 데이터 형식 이해

### Preference 데이터의 기본 구조

선호도 데이터는 동일한 프롬프트에 대해 **"선호 응답(chosen)"**과 **"비선호 응답(rejected)"**을 쌍으로 구성합니다.

```json
{
    "prompt": "질문 또는 지시",
    "chosen": "더 좋은 응답 (인간이 선호)",
    "rejected": "덜 좋은 응답 (인간이 비선호)"
}
```

### 다양한 데이터 형식

| 형식 | 라이브러리 | 설명 |
|------|-----------|------|
| 기본 형식 | - | prompt + chosen + rejected |
| 대화 형식 | TRL DPOTrainer | messages 기반 (system/user/assistant) |
| HF Dataset | datasets | Arrow 포맷 |
| UltraFeedback | - | multi-turn + 점수 기반 |

In [ ]:
# Preference 데이터 형식 예시
print("=" * 60)
print("📌 Preference 데이터 형식 예시")
print("=" * 60)

# 형식 1: 기본 형식 (prompt, chosen, rejected)
basic_format = {
    "prompt": "파이썬에서 리스트의 중복을 제거하는 방법을 알려주세요.",
    "chosen": "파이썬에서 리스트의 중복을 제거하는 방법은 여러 가지가 있습니다.\n\n"
              "1. **set() 사용**: `list(set(my_list))` - 가장 간단하지만 순서가 보장되지 않습니다.\n"
              "2. **dict.fromkeys()**: `list(dict.fromkeys(my_list))` - 순서를 유지합니다.\n"
              "3. **리스트 컴프리헨션**: 순서를 유지하면서 중복 제거가 가능합니다.\n\n"
              "```python\nmy_list = [1, 2, 2, 3, 3, 4]\n"
              "unique = list(dict.fromkeys(my_list))  # [1, 2, 3, 4]\n```",
    "rejected": "set을 쓰면 됩니다."
}

print("\n📋 형식 1: 기본 형식")
print(f"   Prompt: {basic_format['prompt']}")
print(f"   Chosen (길이: {len(basic_format['chosen'])}자): {basic_format['chosen'][:80]}...")
print(f"   Rejected (길이: {len(basic_format['rejected'])}자): {basic_format['rejected']}")

print("\n💡 Chosen이 더 상세하고, 코드 예시를 포함하며, 여러 방법을 소개합니다.")
print("   Rejected는 너무 짧고 불친절합니다.")

In [ ]:
# 형식 2: 대화 형식 (TRL DPOTrainer 호환)
print("=" * 60)
print("📌 형식 2: 대화(Messages) 형식 - TRL DPOTrainer 호환")
print("=" * 60)

conversation_format = {
    "chosen": [
        {"role": "system", "content": "당신은 친절한 AI 어시스턴트입니다."},
        {"role": "user", "content": "파이썬에서 리스트 중복 제거 방법은?"},
        {"role": "assistant", "content": "파이썬에서 리스트 중복을 제거하는 방법은 여러 가지입니다.\n\n"
                                          "1. set() 사용: list(set(my_list))\n"
                                          "2. dict.fromkeys(): 순서 유지\n\n"
                                          "상황에 맞는 방법을 선택하시면 됩니다!"}
    ],
    "rejected": [
        {"role": "system", "content": "당신은 친절한 AI 어시스턴트입니다."},
        {"role": "user", "content": "파이썬에서 리스트 중복 제거 방법은?"},
        {"role": "assistant", "content": "set을 쓰면 됩니다."}
    ]
}

print("\n📋 Chosen 대화:")
for msg in conversation_format["chosen"]:
    print(f"   [{msg['role']}]: {msg['content'][:60]}..." if len(msg['content']) > 60 else f"   [{msg['role']}]: {msg['content']}")

print("\n📋 Rejected 대화:")
for msg in conversation_format["rejected"]:
    print(f"   [{msg['role']}]: {msg['content']}")

print("\n💡 TRL의 DPOTrainer는 이 대화 형식을 직접 지원합니다.")
print("   chosen과 rejected의 system/user 메시지는 동일해야 합니다.")

In [ ]:
# 선호도 판단 기준 정리
print("=" * 60)
print("📌 Chosen vs Rejected 판단 기준")
print("=" * 60)

criteria = [
    {"기준": "정확성 (Accuracy)",
     "chosen": "사실에 기반한 정확한 정보",
     "rejected": "부정확하거나 환각(hallucination) 포함"},
    {"기준": "유용성 (Helpfulness)",
     "chosen": "질문에 대한 완전한 답변",
     "rejected": "불완전하거나 관련 없는 답변"},
    {"기준": "안전성 (Safety)",
     "chosen": "유해하지 않은 안전한 답변",
     "rejected": "편향적이거나 유해한 내용 포함"},
    {"기준": "형식/가독성 (Format)",
     "chosen": "잘 구조화된 마크다운, 코드 블록",
     "rejected": "형식 없는 긴 텍스트"},
    {"기준": "친절함 (Tone)",
     "chosen": "정중하고 교육적인 톤",
     "rejected": "무례하거나 귀찮은 듯한 톤"},
]

for c in criteria:
    print(f"\n📊 {c['기준']}")
    print(f"   ✅ Chosen: {c['chosen']}")
    print(f"   ❌ Rejected: {c['rejected']}")

---
## 2️⃣ 인간 평가 기반 데이터 수집 방법

### 전통적 방법: Human Annotator

```
┌──────────────────────────────────────────┐
│         인간 평가 기반 수집 파이프라인        │
├──────────────────────────────────────────┤
│                                          │
│  1. 프롬프트 수집                          │
│     └→ 실제 사용자 질문 / 합성 프롬프트      │
│                                          │
│  2. 응답 생성 (2개 이상)                    │
│     ├→ 같은 모델로 여러 번 생성             │
│     ├→ 다른 모델들로 각각 생성              │
│     └→ 인간이 직접 작성 + 모델 생성         │
│                                          │
│  3. 인간 평가자가 순위 매김                  │
│     └→ A > B 또는 A = B                   │
│                                          │
│  4. 품질 검수 (합의율, 일관성)               │
│                                          │
│  5. Chosen/Rejected 쌍 구성               │
└──────────────────────────────────────────┘
```

### 비용과 한계

| 항목 | 현실 |
|------|------|
| 비용 | 1건당 $0.5~$2 (크라우드소싱) |
| 시간 | 1,000건 = 수일 |
| 일관성 | 평가자 간 합의율 60-80% |
| 규모 | ChatGPT는 100K+ 데이터 사용 |

→ **현실적 대안: AI를 활용한 자동 생성!**

In [ ]:
# 인간 평가 시뮬레이션
print("=" * 60)
print("📌 인간 평가 시뮬레이션: 어느 응답이 더 좋은가?")
print("=" * 60)

evaluation_examples = [
    {
        "prompt": "머신러닝과 딥러닝의 차이점을 설명해주세요.",
        "response_a": (
            "머신러닝은 데이터에서 패턴을 학습하는 AI의 하위 분야입니다.\n"
            "딥러닝은 머신러닝의 한 기법으로, 신경망을 여러 층으로 쌓아 복잡한 패턴을 학습합니다.\n\n"
            "주요 차이:\n"
            "1. 구조: ML은 다양한 알고리즘, DL은 신경망 기반\n"
            "2. 데이터: DL은 대규모 데이터 필요\n"
            "3. 특성 추출: ML은 수동, DL은 자동"
        ),
        "response_b": (
            "머신러닝이랑 딥러닝은 비슷한건데요, "
            "딥러닝이 더 최신 기술이라서 더 좋습니다. "
            "요즘은 다 딥러닝 씁니다."
        ),
        "better": "A",
        "reason": "A가 구조적이고 정확하며, 구체적인 차이점을 제시"
    },
    {
        "prompt": "건강한 아침 식사 메뉴를 추천해주세요.",
        "response_a": "오트밀이나 과일 먹으세요.",
        "response_b": (
            "건강한 아침 식사 메뉴를 추천드립니다:\n\n"
            "🥣 오트밀 + 블루베리 + 견과류: 식이섬유와 항산화 성분이 풍부\n"
            "🥚 삶은 달걀 + 통밀빵 + 아보카도: 단백질과 건강한 지방\n"
            "🥗 그릭요거트 + 그래놀라 + 바나나: 프로바이오틱스와 에너지\n\n"
            "아침은 하루의 에너지원이므로, 탄수화물+단백질+지방을 균형있게 섭취하세요!"
        ),
        "better": "B",
        "reason": "B가 더 상세하고, 여러 옵션과 영양 정보를 포함"
    }
]

for i, ex in enumerate(evaluation_examples, 1):
    print(f"\n{'='*50}")
    print(f"📝 예시 {i}: {ex['prompt']}")
    print(f"{'='*50}")
    print(f"\n🅰️ 응답 A:")
    print(f"   {ex['response_a'][:100]}..." if len(ex['response_a']) > 100 else f"   {ex['response_a']}")
    print(f"\n🅱️ 응답 B:")
    print(f"   {ex['response_b'][:100]}..." if len(ex['response_b']) > 100 else f"   {ex['response_b']}")
    print(f"\n✅ 더 좋은 응답: {ex['better']}")
    print(f"   이유: {ex['reason']}")

---
## 3️⃣ GPT-4를 활용한 자동 선호도 데이터 생성

### AI-as-a-Judge 방식

강력한 LLM(GPT-4, Claude 등)을 활용하여 자동으로 선호도 데이터를 생성할 수 있습니다.

### 두 가지 접근법

```
방법 1: 응답 생성 + 자동 평가
  1. 프롬프트에 대해 2개 응답 생성 (다른 온도/모델)
  2. GPT-4가 어느 응답이 더 좋은지 판단
  3. chosen/rejected 쌍 구성

방법 2: 의도적 품질 차이 생성
  1. GPT-4로 고품질 응답 생성 → chosen
  2. 의도적으로 품질이 낮은 응답 생성 → rejected
  3. 또는 약한 모델(1.5B)의 응답을 rejected로 사용
```

### 장점
- 🟢 빠른 대규모 데이터 생성
- 🟢 일관된 품질 기준
- 🟢 비용 효율적 (인간 평가 대비)

### 주의점
- 🔴 GPT-4의 편향이 데이터에 반영될 수 있음
- 🔴 지나치게 긴 응답을 선호하는 경향
- 🔴 자기 자신의 스타일을 선호하는 경향

In [ ]:
# 선호도 데이터 자동 생성을 위한 프롬프트 템플릿
print("=" * 60)
print("📌 선호도 데이터 생성 프롬프트 템플릿")
print("=" * 60)

# 방법 2: 의도적 품질 차이 생성 프롬프트
chosen_prompt_template = """당신은 한국어 AI 어시스턴트입니다.
다음 질문에 대해 최고 품질의 답변을 작성하세요.

요구사항:
- 정확하고 사실에 기반한 답변
- 구조적이고 읽기 쉬운 형식 (번호, 소제목 활용)
- 구체적인 예시나 코드 포함
- 친절하고 교육적인 톤
- 적절한 길이 (너무 짧거나 길지 않게)

질문: {question}"""

rejected_prompt_template = """당신은 AI 어시스턴트입니다.
다음 질문에 대해 답변하되, 아래 조건으로 답변하세요:

조건 (의도적으로 품질을 낮춤):
- 매우 짧고 불친절하게
- 구체적인 예시 없이
- 형식 없이 한 문단으로
- 약간의 부정확한 정보 포함 가능

질문: {question}"""

print("\n🟢 Chosen 생성 프롬프트:")
print(chosen_prompt_template[:200] + "...")
print("\n🔴 Rejected 생성 프롬프트:")
print(rejected_prompt_template[:200] + "...")

print("\n💡 핵심: 같은 질문에 대해 의도적으로 다른 품질의 응답을 생성합니다.")

---
## 4️⃣ 실습: OpenAI API로 Preference 데이터 생성

실제로 OpenAI API를 사용하여 선호도 데이터를 자동 생성해보겠습니다.

### 전략
1. 한국어 질문 목록 준비
2. GPT-4o-mini로 **고품질 응답(chosen)** 생성
3. 동일 모델로 **저품질 응답(rejected)** 생성 (프롬프트로 품질 조절)
4. JSON 형식으로 저장

In [ ]:
# 질문 목록 준비
print("=" * 60)
print("📌 선호도 데이터 생성용 질문 목록")
print("=" * 60)

questions = [
    "파이썬의 리스트와 튜플의 차이점을 설명해주세요.",
    "좋은 코드 리뷰를 하려면 어떻게 해야 하나요?",
    "머신러닝 모델의 과적합을 방지하는 방법은?",
    "REST API와 GraphQL의 차이점은 무엇인가요?",
    "깃(Git)에서 브랜치 전략을 세울 때 고려할 점은?",
    "도커(Docker)를 사용하는 이유와 장점은?",
    "데이터베이스 인덱스가 왜 중요한가요?",
    "효과적인 프롬프트 엔지니어링 기법은?",
    "파이썬에서 비동기 프로그래밍의 장점은?",
    "좋은 API 설계 원칙은 무엇인가요?",
]

for i, q in enumerate(questions, 1):
    print(f"   {i:2d}. {q}")

print(f"\n📊 총 {len(questions)}개 질문 준비 완료")

In [ ]:
# OpenAI API를 사용한 선호도 데이터 생성 함수
from openai import OpenAI

client = OpenAI()

def generate_chosen_response(question: str) -> str:
    """고품질 응답(chosen) 생성"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": (
                "당신은 전문적이고 친절한 한국어 AI 어시스턴트입니다. "
                "질문에 대해 정확하고 구조적인 답변을 제공합니다. "
                "적절한 예시와 설명을 포함하세요."
            )},
            {"role": "user", "content": question}
        ],
        temperature=0.7,
        max_tokens=500
    )
    return response.choices[0].message.content


def generate_rejected_response(question: str) -> str:
    """저품질 응답(rejected) 생성"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": (
                "당신은 AI 어시스턴트입니다. "
                "질문에 대해 매우 간략하고 불친절하게 답변하세요. "
                "구체적인 예시 없이 한두 문장으로만 답하세요. "
                "형식적인 구조 없이 답변하세요."
            )},
            {"role": "user", "content": question}
        ],
        temperature=1.0,
        max_tokens=100
    )
    return response.choices[0].message.content


print("✅ 선호도 데이터 생성 함수 정의 완료")
print("   • generate_chosen_response(): 고품질 응답 생성")
print("   • generate_rejected_response(): 저품질 응답 생성")

In [ ]:
# 선호도 데이터 생성 실행
import time

print("=" * 60)
print("📌 선호도 데이터 생성 시작")
print("=" * 60)

preference_data = []

for i, question in enumerate(questions, 1):
    print(f"\n🔄 [{i}/{len(questions)}] 생성 중: {question[:40]}...")
    
    try:
        # 고품질 응답 생성
        chosen = generate_chosen_response(question)
        print(f"   ✅ Chosen 생성 완료 ({len(chosen)}자)")
        
        # 저품질 응답 생성
        rejected = generate_rejected_response(question)
        print(f"   ✅ Rejected 생성 완료 ({len(rejected)}자)")
        
        # 데이터 저장
        preference_data.append({
            "prompt": question,
            "chosen": chosen,
            "rejected": rejected,
            "chosen_model": "gpt-4o-mini (high quality prompt)",
            "rejected_model": "gpt-4o-mini (low quality prompt)",
        })
        
        # Rate limit 방지
        time.sleep(0.5)
        
    except Exception as e:
        print(f"   ❌ 에러: {e}")
        continue

print(f"\n{'='*60}")
print(f"✅ 총 {len(preference_data)}개 선호도 데이터 생성 완료!")

In [ ]:
# 생성된 데이터 확인
print("=" * 60)
print("📌 생성된 선호도 데이터 미리보기")
print("=" * 60)

for i, item in enumerate(preference_data[:3], 1):
    print(f"\n{'='*50}")
    print(f"📝 데이터 #{i}")
    print(f"{'='*50}")
    print(f"\n❓ Prompt: {item['prompt']}")
    print(f"\n✅ Chosen ({len(item['chosen'])}자):")
    print(f"   {item['chosen'][:200]}..." if len(item['chosen']) > 200 else f"   {item['chosen']}")
    print(f"\n❌ Rejected ({len(item['rejected'])}자):")
    print(f"   {item['rejected'][:200]}..." if len(item['rejected']) > 200 else f"   {item['rejected']}")

if not preference_data:
    print("\n⚠️ 데이터가 생성되지 않았습니다. API 키를 확인해주세요.")
    print("   → 다음 섹션에서 샘플 데이터를 사용하겠습니다.")

In [ ]:
# 생성된 데이터를 JSON으로 저장
output_path = OUTPUT_DIR / "generated_preference_data.json"

if preference_data:
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(preference_data, f, ensure_ascii=False, indent=2)
    print(f"✅ 데이터 저장 완료: {output_path}")
    print(f"   총 {len(preference_data)}개 샘플")
else:
    print("⚠️ 저장할 데이터가 없습니다.")

---
## 5️⃣ 샘플 데이터 확인

API를 사용하지 못하는 경우를 대비하여 미리 준비된 샘플 데이터를 확인합니다.  
이 데이터는 다음 세션(Session 28: DPO 학습)에서도 사용됩니다.

In [ ]:
# 샘플 선호도 데이터 생성 (API 없이도 사용 가능)
print("=" * 60)
print("📌 샘플 Preference 데이터 생성")
print("=" * 60)

sample_preference_data = [
    {
        "prompt": "파이썬에서 리스트와 튜플의 차이점을 설명해주세요.",
        "chosen": (
            "파이썬에서 리스트(list)와 튜플(tuple)은 모두 순서가 있는 컬렉션이지만, "
            "중요한 차이점이 있습니다.\n\n"
            "1. **변경 가능성(Mutability)**\n"
            "   - 리스트: 변경 가능(mutable) - 요소 추가/삭제/수정 가능\n"
            "   - 튜플: 변경 불가(immutable) - 생성 후 수정 불가\n\n"
            "2. **문법**\n"
            "   - 리스트: `[1, 2, 3]` (대괄호)\n"
            "   - 튜플: `(1, 2, 3)` (소괄호)\n\n"
            "3. **성능**\n"
            "   - 튜플이 리스트보다 약간 빠르고 메모리를 적게 사용합니다.\n\n"
            "4. **사용 시점**\n"
            "   - 리스트: 데이터가 변경될 수 있을 때\n"
            "   - 튜플: 변경되지 않아야 하는 데이터 (딕셔너리 키 등)\n\n"
            "```python\n"
            "# 리스트 - 변경 가능\n"
            "my_list = [1, 2, 3]\n"
            "my_list[0] = 10  # OK\n\n"
            "# 튜플 - 변경 불가\n"
            "my_tuple = (1, 2, 3)\n"
            "# my_tuple[0] = 10  # TypeError!\n"
            "```"
        ),
        "rejected": "리스트는 대괄호고 튜플은 소괄호입니다. 리스트는 바꿀 수 있고 튜플은 못 바꿉니다."
    },
    {
        "prompt": "좋은 코드 리뷰를 하려면 어떻게 해야 하나요?",
        "chosen": (
            "좋은 코드 리뷰를 위한 핵심 원칙과 실천 방법을 소개합니다.\n\n"
            "**1. 마인드셋**\n"
            "- 코드를 비판하되, 사람을 비판하지 마세요\n"
            "- '이것은 틀렸다' 대신 '이렇게 하면 어떨까요?'로 제안\n\n"
            "**2. 확인할 항목**\n"
            "- 로직의 정확성과 엣지 케이스\n"
            "- 코드 가독성과 네이밍\n"
            "- 테스트 커버리지\n"
            "- 보안 취약점\n"
            "- 성능 이슈\n\n"
            "**3. 실천 팁**\n"
            "- PR 크기를 작게 유지 (200줄 이하 권장)\n"
            "- 24시간 이내 리뷰 완료\n"
            "- 긍정적 피드백도 함께 남기기"
        ),
        "rejected": "코드를 잘 읽고 문제 있으면 말하면 됩니다."
    },
    {
        "prompt": "머신러닝 모델의 과적합을 방지하는 방법은?",
        "chosen": (
            "과적합(Overfitting)은 모델이 학습 데이터에 지나치게 맞춰져 "
            "새로운 데이터에 일반화하지 못하는 현상입니다. 주요 방지 방법은:\n\n"
            "**1. 데이터 측면**\n"
            "- 더 많은 학습 데이터 수집\n"
            "- 데이터 증강(Data Augmentation)\n\n"
            "**2. 모델 측면**\n"
            "- 정규화: L1/L2 Regularization\n"
            "- 드롭아웃(Dropout) 적용\n"
            "- 모델 복잡도 줄이기\n\n"
            "**3. 학습 과정**\n"
            "- 조기 종료(Early Stopping)\n"
            "- 교차 검증(Cross-Validation)\n"
            "- 학습률 스케줄링\n\n"
            "가장 효과적인 방법은 양질의 데이터를 충분히 확보하는 것입니다."
        ),
        "rejected": "드롭아웃 쓰면 됩니다."
    },
    {
        "prompt": "REST API와 GraphQL의 차이점은 무엇인가요?",
        "chosen": (
            "REST API와 GraphQL은 서로 다른 API 설계 패러다임입니다.\n\n"
            "**REST API**\n"
            "- 리소스 기반 URL 엔드포인트 (예: /users, /posts)\n"
            "- HTTP 메서드 활용 (GET, POST, PUT, DELETE)\n"
            "- 고정된 응답 구조\n"
            "- 여러 엔드포인트 필요할 수 있음\n\n"
            "**GraphQL**\n"
            "- 단일 엔드포인트 (/graphql)\n"
            "- 클라이언트가 필요한 데이터만 요청\n"
            "- Over-fetching/Under-fetching 해결\n"
            "- 강력한 타입 시스템\n\n"
            "**선택 기준**\n"
            "- 간단한 CRUD: REST\n"
            "- 복잡한 데이터 관계: GraphQL\n"
            "- 캐싱 중요: REST (HTTP 캐싱 활용)\n"
            "- 모바일 앱: GraphQL (네트워크 효율)"
        ),
        "rejected": "REST는 여러 엔드포인트 쓰고 GraphQL은 하나만 씁니다. 요즘은 GraphQL이 트렌드입니다."
    },
    {
        "prompt": "깃(Git)에서 브랜치 전략을 세울 때 고려할 점은?",
        "chosen": (
            "Git 브랜치 전략은 팀의 협업 효율성에 직접적인 영향을 미칩니다.\n\n"
            "**주요 브랜치 전략**\n"
            "1. **Git Flow**: main/develop/feature/release/hotfix\n"
            "   - 대규모 프로젝트, 릴리스 주기가 긴 경우\n"
            "2. **GitHub Flow**: main + feature branches\n"
            "   - 지속적 배포, 간단한 구조\n"
            "3. **Trunk-Based**: 짧은 브랜치, 빠른 머지\n"
            "   - CI/CD가 잘 갖춰진 팀\n\n"
            "**고려할 점**\n"
            "- 팀 규모와 경험 수준\n"
            "- 배포 주기 (일별/주별/월별)\n"
            "- 코드 리뷰 프로세스\n"
            "- CI/CD 파이프라인\n"
            "- 핫픽스 대응 방안"
        ),
        "rejected": "main이랑 feature 브랜치 쓰면 됩니다."
    },
    {
        "prompt": "도커(Docker)를 사용하는 이유와 장점은?",
        "chosen": (
            "Docker는 컨테이너 기반 가상화 플랫폼으로, 현대 소프트웨어 개발에서 "
            "필수적인 도구입니다.\n\n"
            "**주요 장점**\n"
            "1. **환경 일관성**: '내 PC에서는 되는데...' 문제 해결\n"
            "2. **빠른 배포**: 이미지 기반으로 몇 초만에 환경 구축\n"
            "3. **격리성**: 프로세스/네트워크/파일시스템 격리\n"
            "4. **확장성**: 오케스트레이션(K8s)과 결합하여 쉬운 스케일링\n"
            "5. **리소스 효율**: VM 대비 가벼움 (OS 커널 공유)\n\n"
            "**대표적 활용 사례**\n"
            "- 개발/테스트 환경 통일\n"
            "- 마이크로서비스 아키텍처\n"
            "- CI/CD 파이프라인\n"
            "- ML 모델 서빙"
        ),
        "rejected": "환경 설정 귀찮을 때 씁니다. 가상 머신보다 가볍습니다."
    },
    {
        "prompt": "데이터베이스 인덱스가 왜 중요한가요?",
        "chosen": (
            "데이터베이스 인덱스는 **쿼리 성능을 극적으로 향상**시키는 데이터 구조입니다.\n\n"
            "**인덱스의 원리**\n"
            "- 책의 색인처럼, 원하는 데이터의 위치를 빠르게 찾음\n"
            "- B-Tree, Hash, GIN 등 다양한 구조\n\n"
            "**성능 차이 예시**\n"
            "- 100만 건 테이블에서 특정 행 검색:\n"
            "  - 인덱스 없음: Full Table Scan → O(n) = ~수 초\n"
            "  - 인덱스 있음: B-Tree 탐색 → O(log n) = ~수 ms\n\n"
            "**주의사항**\n"
            "- INSERT/UPDATE 시 인덱스 유지 비용 발생\n"
            "- 과도한 인덱스는 오히려 성능 저하\n"
            "- WHERE, JOIN, ORDER BY에 사용되는 컬럼에 생성"
        ),
        "rejected": "쿼리가 빨라집니다."
    },
    {
        "prompt": "효과적인 프롬프트 엔지니어링 기법은?",
        "chosen": (
            "프롬프트 엔지니어링은 LLM에서 원하는 결과를 얻기 위한 핵심 기술입니다.\n\n"
            "**기본 기법**\n"
            "1. **역할 부여**: '당신은 시니어 개발자입니다'\n"
            "2. **구체적 지시**: 모호한 표현 대신 명확한 요구사항\n"
            "3. **출력 형식 지정**: JSON, 표, 번호 목록 등\n\n"
            "**고급 기법**\n"
            "4. **Few-shot**: 예시를 2-3개 제공\n"
            "5. **Chain-of-Thought**: '단계별로 생각해보세요'\n"
            "6. **Self-Consistency**: 여러 번 생성 후 다수결\n\n"
            "**실전 팁**\n"
            "- 구분자(```, ###)로 입력/출력 영역 구분\n"
            "- 부정문보다 긍정문 사용\n"
            "- 반복적으로 프롬프트를 개선(iteration)"
        ),
        "rejected": "잘 물어보면 됩니다. 구체적으로 쓰세요."
    },
    {
        "prompt": "파이썬에서 비동기 프로그래밍의 장점은?",
        "chosen": (
            "파이썬의 비동기 프로그래밍(async/await)은 I/O 바운드 작업의 "
            "효율성을 크게 향상시킵니다.\n\n"
            "**핵심 장점**\n"
            "1. **높은 동시성**: 대기 시간 동안 다른 작업 처리\n"
            "2. **리소스 효율**: 스레드 대비 메모리 사용량 적음\n"
            "3. **확장성**: 수천 개의 동시 연결 처리 가능\n\n"
            "**적합한 경우**\n"
            "- 웹 서버 (FastAPI, aiohttp)\n"
            "- API 호출 다수\n"
            "- 파일/DB I/O 작업\n\n"
            "**부적합한 경우**\n"
            "- CPU 바운드 작업 (수학 계산 등)\n"
            "- 간단한 스크립트\n\n"
            "```python\n"
            "import asyncio\n\n"
            "async def fetch_data():\n"
            "    await asyncio.sleep(1)  # I/O 대기\n"
            "    return 'data'\n"
            "```"
        ),
        "rejected": "동시에 여러 작업을 할 수 있어서 빠릅니다."
    },
    {
        "prompt": "좋은 API 설계 원칙은 무엇인가요?",
        "chosen": (
            "좋은 API 설계는 사용자 경험과 유지보수성을 결정합니다.\n\n"
            "**핵심 원칙**\n"
            "1. **일관성**: 네이밍, 에러 형식, 페이징 등 통일\n"
            "2. **직관적 URL**: `/users/{id}/posts` 처럼 리소스 중심\n"
            "3. **적절한 HTTP 상태 코드**: 200, 201, 400, 404, 500\n"
            "4. **버전 관리**: `/api/v1/...`\n"
            "5. **페이징**: 대량 데이터 반환 시 필수\n\n"
            "**추가 고려사항**\n"
            "- 인증/인가 (OAuth 2.0, JWT)\n"
            "- Rate Limiting\n"
            "- 명확한 에러 메시지\n"
            "- API 문서화 (OpenAPI/Swagger)\n"
            "- 하위 호환성 유지"
        ),
        "rejected": "RESTful하게 만들면 됩니다."
    }
]

# 샘플 데이터 저장
sample_path = DATA_DIR / "preference_sample.json"
with open(sample_path, "w", encoding="utf-8") as f:
    json.dump(sample_preference_data, f, ensure_ascii=False, indent=2)

print(f"✅ 샘플 데이터 저장 완료: {sample_path}")
print(f"   총 {len(sample_preference_data)}개 샘플")

In [ ]:
# 저장된 샘플 데이터 로드 및 확인
print("=" * 60)
print("📌 저장된 샘플 데이터 로드 및 확인")
print("=" * 60)

sample_path = DATA_DIR / "preference_sample.json"

with open(sample_path, "r", encoding="utf-8") as f:
    loaded_data = json.load(f)

print(f"\n📦 로드된 데이터: {len(loaded_data)}개 샘플")
print(f"📋 데이터 키: {list(loaded_data[0].keys())}")

# 통계
chosen_lengths = [len(d["chosen"]) for d in loaded_data]
rejected_lengths = [len(d["rejected"]) for d in loaded_data]

print(f"\n📊 Chosen 응답 길이:")
print(f"   평균: {sum(chosen_lengths)/len(chosen_lengths):.0f}자")
print(f"   최소: {min(chosen_lengths)}자, 최대: {max(chosen_lengths)}자")

print(f"\n📊 Rejected 응답 길이:")
print(f"   평균: {sum(rejected_lengths)/len(rejected_lengths):.0f}자")
print(f"   최소: {min(rejected_lengths)}자, 최대: {max(rejected_lengths)}자")

print(f"\n📊 Chosen/Rejected 길이 비율: {sum(chosen_lengths)/sum(rejected_lengths):.1f}배")

---
## 6️⃣ 데이터 품질 검증

선호도 데이터의 품질은 DPO 학습 성능에 직접적인 영향을 미칩니다.  
체계적인 품질 검증 방법을 실습합니다.

In [ ]:
# 데이터 품질 검증 함수
print("=" * 60)
print("📌 선호도 데이터 품질 검증")
print("=" * 60)

def validate_preference_data(data: list) -> dict:
    """
    선호도 데이터의 품질을 검증합니다.
    """
    issues = []
    stats = {
        "total": len(data),
        "valid": 0,
        "warnings": 0,
        "errors": 0,
    }
    
    for i, item in enumerate(data):
        item_issues = []
        
        # 1. 필수 키 확인
        for key in ["prompt", "chosen", "rejected"]:
            if key not in item:
                item_issues.append(f"❌ 필수 키 '{key}' 누락")
                stats["errors"] += 1
        
        if item_issues:
            issues.append((i, item_issues))
            continue
        
        # 2. 빈 문자열 체크
        for key in ["prompt", "chosen", "rejected"]:
            if not item[key].strip():
                item_issues.append(f"❌ '{key}'가 비어있음")
                stats["errors"] += 1
        
        # 3. Chosen이 Rejected보다 짧은 경우 (경고)
        if len(item["chosen"]) < len(item["rejected"]):
            item_issues.append(f"⚠️ Chosen({len(item['chosen'])}자)이 Rejected({len(item['rejected'])}자)보다 짧음")
            stats["warnings"] += 1
        
        # 4. Chosen과 Rejected가 동일한 경우
        if item["chosen"].strip() == item["rejected"].strip():
            item_issues.append("❌ Chosen과 Rejected가 동일함")
            stats["errors"] += 1
        
        # 5. 너무 짧은 응답 (경고)
        if len(item["chosen"]) < 20:
            item_issues.append(f"⚠️ Chosen이 너무 짧음 ({len(item['chosen'])}자)")
            stats["warnings"] += 1
        
        # 6. 너무 긴 응답 (경고)
        if len(item["chosen"]) > 2000:
            item_issues.append(f"⚠️ Chosen이 너무 김 ({len(item['chosen'])}자)")
            stats["warnings"] += 1
        
        if item_issues:
            issues.append((i, item_issues))
        else:
            stats["valid"] += 1
    
    return stats, issues

# 검증 실행
stats, issues = validate_preference_data(loaded_data)

print(f"\n📊 검증 결과:")
print(f"   총 데이터: {stats['total']}개")
print(f"   ✅ 유효: {stats['valid']}개")
print(f"   ⚠️ 경고: {stats['warnings']}개")
print(f"   ❌ 오류: {stats['errors']}개")

if issues:
    print(f"\n📋 상세 이슈:")
    for idx, item_issues in issues:
        print(f"   데이터 #{idx}:")
        for issue in item_issues:
            print(f"      {issue}")
else:
    print("\n✅ 모든 데이터가 품질 검증을 통과했습니다!")

In [ ]:
# DPO 학습에 적합한 형식으로 변환
print("=" * 60)
print("📌 DPO 학습용 데이터 형식 변환")
print("=" * 60)

def convert_to_dpo_format(data: list, system_prompt: str = "당신은 친절하고 도움이 되는 AI 어시스턴트입니다.") -> list:
    """
    기본 형식의 preference 데이터를 TRL DPOTrainer 형식으로 변환합니다.
    """
    dpo_data = []
    
    for item in data:
        dpo_item = {
            "chosen": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": item["prompt"]},
                {"role": "assistant", "content": item["chosen"]}
            ],
            "rejected": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": item["prompt"]},
                {"role": "assistant", "content": item["rejected"]}
            ]
        }
        dpo_data.append(dpo_item)
    
    return dpo_data

# 변환 실행
dpo_formatted = convert_to_dpo_format(loaded_data)

# 변환 결과 확인
print(f"\n✅ 변환 완료: {len(dpo_formatted)}개 샘플")
print(f"\n📋 변환된 데이터 예시 (첫 번째):")
print(json.dumps(dpo_formatted[0], ensure_ascii=False, indent=2)[:500] + "...")

# DPO 형식으로 저장
dpo_path = DATA_DIR / "preference_dpo_format.json"
with open(dpo_path, "w", encoding="utf-8") as f:
    json.dump(dpo_formatted, f, ensure_ascii=False, indent=2)

print(f"\n✅ DPO 형식 데이터 저장: {dpo_path}")

---
## 7️⃣ Qwen2.5-1.5B로 Rejected 응답 생성 (Optional GPU)

더 현실적인 선호도 데이터를 만들기 위해, 로컬 모델(Qwen2.5-1.5B)의 출력을 rejected로 사용할 수 있습니다.  
이 방법은 **실제 학습할 모델의 출력**을 rejected로 사용하므로, DPO 학습에 더 효과적입니다.

### 전략
- Chosen: GPT-4o-mini의 고품질 응답
- Rejected: Qwen2.5-1.5B의 (상대적으로 낮은 품질) 응답

> ⚠️ GPU가 없는 경우 이 섹션을 건너뛰세요.

In [ ]:
# GPU 확인 및 모델 로드
import torch
import gc

def print_gpu_memory(tag=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")

print("=" * 60)
print("📌 GPU 환경 확인")
print("=" * 60)

if torch.cuda.is_available():
    print(f"✅ GPU 사용 가능: {torch.cuda.get_device_name(0)}")
    print_gpu_memory("초기 상태")
    GPU_AVAILABLE = True
else:
    print("⚠️ GPU를 사용할 수 없습니다. 이 섹션을 건너뛰세요.")
    GPU_AVAILABLE = False

In [ ]:
# Qwen2.5-1.5B 로드 (GPU 사용 가능 시)
if GPU_AVAILABLE:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    
    model_name = "Qwen/Qwen2.5-1.5B-Instruct"
    
    print(f"🔄 모델 로딩: {model_name}")
    
    # 4bit 양자화 설정
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
    )
    
    print(f"✅ 모델 로딩 완료")
    print_gpu_memory("모델 로드 후")
else:
    print("⏭️ GPU가 없으므로 건너뜁니다.")

In [ ]:
# 로컬 모델로 rejected 응답 생성
if GPU_AVAILABLE:
    print("=" * 60)
    print("📌 Qwen2.5-1.5B로 Rejected 응답 생성")
    print("=" * 60)
    
    def generate_local_response(prompt: str, max_new_tokens: int = 256) -> str:
        """로컬 모델로 응답 생성"""
        messages = [
            {"role": "system", "content": "당신은 AI 어시스턴트입니다."},
            {"role": "user", "content": prompt}
        ]
        
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.7,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
        
        response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        return response.strip()
    
    # 테스트
    test_prompt = loaded_data[0]["prompt"]
    print(f"\n📝 테스트 프롬프트: {test_prompt}")
    
    local_response = generate_local_response(test_prompt)
    print(f"\n🤖 Qwen2.5-1.5B 응답:")
    print(f"   {local_response[:300]}..." if len(local_response) > 300 else f"   {local_response}")
    
    print_gpu_memory("생성 후")
else:
    print("⏭️ GPU가 없으므로 건너뜁니다.")

In [ ]:
# 로컬 모델 기반 preference 데이터 생성 (일부만)
if GPU_AVAILABLE:
    print("=" * 60)
    print("📌 로컬 모델 기반 Preference 데이터 생성")
    print("=" * 60)
    
    local_preference_data = []
    
    # 처음 3개만 생성 (시간 절약)
    for i, item in enumerate(loaded_data[:3], 1):
        print(f"\n🔄 [{i}/3] 생성 중: {item['prompt'][:40]}...")
        
        # 로컬 모델의 응답을 rejected로 사용
        local_response = generate_local_response(item['prompt'])
        
        local_preference_data.append({
            "prompt": item["prompt"],
            "chosen": item["chosen"],  # 기존 고품질 응답 유지
            "rejected": local_response,  # 로컬 모델 출력을 rejected로
            "chosen_source": "gpt-4o-mini",
            "rejected_source": "Qwen2.5-1.5B-Instruct",
        })
        
        print(f"   ✅ Rejected 생성 완료 ({len(local_response)}자)")
    
    # 저장
    local_pref_path = OUTPUT_DIR / "local_model_preference_data.json"
    with open(local_pref_path, "w", encoding="utf-8") as f:
        json.dump(local_preference_data, f, ensure_ascii=False, indent=2)
    
    print(f"\n✅ 로컬 모델 기반 데이터 저장: {local_pref_path}")
    print(f"   총 {len(local_preference_data)}개 샘플")
    
    print_gpu_memory("데이터 생성 완료")
else:
    print("⏭️ GPU가 없으므로 건너뜁니다.")
    print("   → 다음 세션에서 샘플 데이터를 사용하여 DPO 학습을 진행합니다.")

In [ ]:
# GPU 메모리 정리
if GPU_AVAILABLE:
    print("🧹 GPU 메모리 정리 중...")
    print_gpu_memory("정리 전")
    
    if 'model' in dir():
        del model
    if 'tokenizer' in dir():
        del tokenizer
    
    gc.collect()
    torch.cuda.empty_cache()
    
    print_gpu_memory("정리 후")
    print("✅ GPU 메모리 정리 완료")

---
## 📝 정리 및 핵심 요약

### 이번 세션에서 배운 내용

1️⃣ **선호도 데이터 구조**: prompt + chosen + rejected 쌍

2️⃣ **TRL DPOTrainer 형식**: messages 기반 대화 형식으로 변환

3️⃣ **인간 평가 기반 수집**: 비용이 높지만 가장 정확

4️⃣ **AI 기반 자동 생성**: GPT-4로 의도적 품질 차이를 만들어 대규모 생성

5️⃣ **로컬 모델 활용**: 학습 대상 모델의 출력을 rejected로 사용

6️⃣ **품질 검증**: 필수 키 확인, 길이 검증, 내용 중복 체크

### 생성된 파일 정리

| 파일 | 용도 |
|------|------|
| `../data/samples/preference_sample.json` | 기본 형식 샘플 데이터 |
| `../data/samples/preference_dpo_format.json` | DPO 학습용 형식 데이터 |
| `./outputs/generated_preference_data.json` | API로 생성한 데이터 |

### 다음 세션 예고

- 📘 **Session 28**: 이 데이터를 사용하여 **DPO 학습** 실습 (SFT → DPO)

In [ ]:
# 최종 파일 확인
print("=" * 60)
print("📌 생성된 파일 확인")
print("=" * 60)

files_to_check = [
    DATA_DIR / "preference_sample.json",
    DATA_DIR / "preference_dpo_format.json",
    OUTPUT_DIR / "generated_preference_data.json",
]

for fpath in files_to_check:
    if fpath.exists():
        size_kb = fpath.stat().st_size / 1024
        with open(fpath, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"  ✅ {fpath.name}: {len(data)}개 샘플, {size_kb:.1f}KB")
    else:
        print(f"  ⚠️ {fpath.name}: 파일 없음")

print("\n" + "=" * 60)
print("✅ Session 27 완료! 다음 세션에서 DPO 학습을 진행합니다.")